# Advanced Problems with Solutions — Python Class Attributes

This notebook is a self-contained advanced practice set built around **class attributes**.

## Core concepts practiced

- built-in class attributes such as `__name__`
- user-defined class attributes
- dotted access versus `getattr`
- mutation with dotted assignment and `setattr`
- dynamic creation of class attributes
- deletion with `del` and `delattr`
- the class namespace exposed through `Class.__dict__`
- why `Class.__dict__` is a read-only `mappingproxy`
- class vs. instance lookup
- shadowing
- mutable class-attribute pitfalls
- inheritance and the MRO
- inspecting *local* namespace state versus resolved attributes
- class methods and class-level state
- runtime monkey-patching
- defensive patterns for configuration and registries

The exercises intentionally progress from advanced fundamentals to subtle lookup behavior.

> **Best-practice rule:** before running each solution, predict the result and explain *where Python will look for the attribute*.

## Setup: a tiny inspection helper

`show_class_state` prints only selected entries from a class's own namespace. This makes later exercises easier to inspect without drowning in Python's built-in class metadata.

In [1]:
def show_class_state(cls, *names):
    """Return selected names that live directly in cls.__dict__."""
    return {name: cls.__dict__[name] for name in names if name in cls.__dict__}


def divider(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

# Part I — Class Namespace Mechanics

## Problem 1 — Built-in vs. user-defined class attributes

Create a class `LanguageRuntime` with class attributes `language` and `version`.

Then determine:

1. whether `language` is present directly in `LanguageRuntime.__dict__`
2. whether `__name__` is present directly in `LanguageRuntime.__dict__`
3. whether both are nevertheless accessible with dotted notation
4. what this tells you about the difference between **attribute lookup** and **direct namespace membership**

In [2]:
# Solution 1

class LanguageRuntime:
    language = "Python"
    version = "3.12"

print(LanguageRuntime.language)
print(LanguageRuntime.__name__)

print("language" in LanguageRuntime.__dict__)
print("__name__" in LanguageRuntime.__dict__)

assert LanguageRuntime.language == "Python"
assert LanguageRuntime.__name__ == "LanguageRuntime"
assert "language" in LanguageRuntime.__dict__
assert "__name__" not in LanguageRuntime.__dict__

Python
LanguageRuntime
True
False


### Solution explanation

`LanguageRuntime.__dict__` shows the namespace entries stored directly on the class object. Attribute lookup is broader than that mapping alone. `__name__` is available as an attribute of the class object even though it is not an ordinary entry in the class namespace mapping shown here.

**Key lesson:** "`name in cls.__dict__`" and "`hasattr(cls, name)`" answer different questions.

## Problem 2 — Dotted access, `getattr`, and fallback values

Implement `read_setting(cls, name, default)` so that it safely reads a class attribute without raising `AttributeError` when the name is absent.

Test it on known and unknown names.

In [3]:
# Solution 2

class ServiceConfig:
    host = "localhost"
    port = 8000

def read_setting(cls, name, default=None):
    return getattr(cls, name, default)

print(read_setting(ServiceConfig, "host"))
print(read_setting(ServiceConfig, "port"))
print(read_setting(ServiceConfig, "timeout", 30))

assert read_setting(ServiceConfig, "host") == "localhost"
assert read_setting(ServiceConfig, "timeout", 30) == 30

localhost
8000
30


### Best-practice note

Use `getattr(obj, name, default)` when the attribute name is dynamic. Prefer ordinary dotted access (`obj.name`) when the name is fixed in source code because it is clearer and easier for static-analysis tools to understand.

## Problem 3 — Dynamic class configuration with `setattr`

Given a dictionary of configuration values, add all values to a class at runtime. Verify that the attributes were added to the class namespace.

In [4]:
# Solution 3

class AppConfig:
    pass

settings = {
    "debug": True,
    "workers": 4,
    "region": "eu-central",
    "retry_limit": 5,
}

for name, value in settings.items():
    setattr(AppConfig, name, value)

for name in settings:
    print(name, "->", getattr(AppConfig, name))

print(show_class_state(AppConfig, *settings))

assert all(name in AppConfig.__dict__ for name in settings)
assert AppConfig.workers == 4

debug -> True
workers -> 4
region -> eu-central
retry_limit -> 5
{'debug': True, 'workers': 4, 'region': 'eu-central', 'retry_limit': 5}


## Problem 4 — Reject unsafe dynamic attribute names

Dynamic mutation is powerful, but accepting arbitrary names can overwrite important class behavior.

Write `safe_set_class_attribute(cls, name, value)` that rejects:

- names beginning with `_`
- names already present in the class's own namespace
- names that are not valid Python identifiers

Then test successful and failing cases.

In [5]:
# Solution 4

def safe_set_class_attribute(cls, name, value):
    if not isinstance(name, str) or not name.isidentifier():
        raise ValueError(f"Invalid attribute name: {name!r}")
    if name.startswith("_"):
        raise ValueError("Private/dunder-style names are not allowed")
    if name in cls.__dict__:
        raise ValueError(f"{name!r} already exists directly on {cls.__name__}")
    setattr(cls, name, value)

class FeatureFlags:
    enabled = True

safe_set_class_attribute(FeatureFlags, "max_items", 100)
print(FeatureFlags.max_items)

for bad_name in ["enabled", "__dict__", "not-valid"]:
    try:
        safe_set_class_attribute(FeatureFlags, bad_name, 1)
    except ValueError as exc:
        print(type(exc).__name__ + ":", exc)

assert FeatureFlags.max_items == 100

100
ValueError: 'enabled' already exists directly on FeatureFlags
ValueError: Private/dunder-style names are not allowed
ValueError: Invalid attribute name: 'not-valid'


## Problem 5 — Mutation is reflected by the `mappingproxy`

Create a class attribute, keep a reference to `Class.__dict__`, mutate the class using `setattr`, and prove that the previously obtained mapping proxy reflects the new state.

In [6]:
# Solution 5

class RuntimeState:
    status = "starting"

namespace_view = RuntimeState.__dict__

print("before:", namespace_view["status"])

setattr(RuntimeState, "status", "ready")
setattr(RuntimeState, "workers", 8)

print("after status:", namespace_view["status"])
print("new key visible:", namespace_view["workers"])

assert namespace_view["status"] == "ready"
assert namespace_view["workers"] == 8

before: starting
after status: ready
new key visible: 8


### Solution explanation

A `mappingproxy` is a **read-only view**, not a frozen copy. You cannot assign through it, but changes made to the underlying class namespace through supported class operations become visible through the same proxy.

## Problem 6 — Prove that `mappingproxy` is read-only

Attempt to write directly to a class's `__dict__`. Catch the exception, then perform the equivalent legal mutation using `setattr`.

In [7]:
# Solution 6

class ImmutableViewDemo:
    x = 10

try:
    ImmutableViewDemo.__dict__["x"] = 99
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

setattr(ImmutableViewDemo, "x", 99)
print(ImmutableViewDemo.x)

assert ImmutableViewDemo.x == 99

TypeError: 'mappingproxy' object does not support item assignment
99


## Problem 7 — Delete dynamically created class attributes

Create three runtime class attributes, delete one with `del`, another with `delattr`, and leave the third untouched.

Verify the final class namespace.

In [8]:
# Solution 7

class CachePolicy:
    pass

CachePolicy.ttl = 60
CachePolicy.max_entries = 1000
CachePolicy.strategy = "LRU"

del CachePolicy.ttl
delattr(CachePolicy, "max_entries")

print("ttl present?", "ttl" in CachePolicy.__dict__)
print("max_entries present?", "max_entries" in CachePolicy.__dict__)
print("strategy present?", "strategy" in CachePolicy.__dict__)

assert "ttl" not in CachePolicy.__dict__
assert "max_entries" not in CachePolicy.__dict__
assert CachePolicy.strategy == "LRU"

ttl present? False
max_entries present? False
strategy present? True


## Problem 8 — Build a namespace diff tool

Write a function that captures a class's direct namespace before and after runtime mutations and reports which names were added, removed, or changed.

Ignore double-underscore names in the report.

In [9]:
# Solution 8

def public_namespace_snapshot(cls):
    return {
        key: value
        for key, value in cls.__dict__.items()
        if not key.startswith("__")
    }

def namespace_diff(before, after):
    before_keys = set(before)
    after_keys = set(after)

    added = {k: after[k] for k in sorted(after_keys - before_keys)}
    removed = {k: before[k] for k in sorted(before_keys - after_keys)}
    changed = {
        k: (before[k], after[k])
        for k in sorted(before_keys & after_keys)
        if before[k] != after[k]
    }
    return {"added": added, "removed": removed, "changed": changed}

class Pipeline:
    mode = "batch"
    retries = 2

before = public_namespace_snapshot(Pipeline)

Pipeline.mode = "stream"
Pipeline.timeout = 15
del Pipeline.retries

after = public_namespace_snapshot(Pipeline)
diff = namespace_diff(before, after)

print(diff)

assert diff["added"] == {"timeout": 15}
assert diff["removed"] == {"retries": 2}
assert diff["changed"] == {"mode": ("batch", "stream")}

{'added': {'timeout': 15}, 'removed': {'retries': 2}, 'changed': {'mode': ('batch', 'stream')}}


# Part II — Class vs. Instance Lookup

## Problem 9 — Predict instance access to a class attribute

Create two instances of a class with class attribute `rate = 0.1`.

Change `Class.rate` and observe both instances. Then assign `rate` on only one instance.

Explain the final three values:

- `Class.rate`
- `instance_a.rate`
- `instance_b.rate`

In [10]:
# Solution 9

class TaxRule:
    rate = 0.10

a = TaxRule()
b = TaxRule()

print(a.rate, b.rate)

TaxRule.rate = 0.20
print(a.rate, b.rate)

a.rate = 0.35
print("class:", TaxRule.rate)
print("a:", a.rate)
print("b:", b.rate)

print("a.__dict__:", a.__dict__)
print("b.__dict__:", b.__dict__)

assert TaxRule.rate == 0.20
assert a.rate == 0.35
assert b.rate == 0.20

0.1 0.1
0.2 0.2
class: 0.2
a: 0.35
b: 0.2
a.__dict__: {'rate': 0.35}
b.__dict__: {}


### Solution explanation

When `a.rate` is not stored on `a`, Python can resolve it from the class. Assigning `a.rate = 0.35` creates an **instance attribute** that shadows the class attribute for that object only. It does not change `TaxRule.rate`.

## Problem 10 — Remove an instance shadow

Continue the shadowing idea. Assign an instance attribute with the same name as a class attribute, then delete the instance attribute and show that class-level lookup becomes visible again.

In [11]:
# Solution 10

class Theme:
    color = "blue"

item = Theme()
item.color = "red"

print("shadowed:", item.color)
print("instance namespace:", item.__dict__)

del item.color

print("after deleting shadow:", item.color)
print("instance namespace:", item.__dict__)

assert item.color == "blue"
assert "color" not in item.__dict__

shadowed: red
instance namespace: {'color': 'red'}
after deleting shadow: blue
instance namespace: {}


## Problem 11 — `getattr` vs. direct namespace membership on instances

For an object whose attribute is inherited from the class, compare:

- `getattr(obj, "name")`
- `"name" in obj.__dict__`
- `"name" in type(obj).__dict__`

Build a helper that reports all three.

In [12]:
# Solution 11

def locate_instance_attribute(obj, name):
    cls = type(obj)
    return {
        "resolved_value": getattr(obj, name),
        "stored_on_instance": name in getattr(obj, "__dict__", {}),
        "stored_on_class": name in cls.__dict__,
    }

class Account:
    currency = "EUR"

acct = Account()
print(locate_instance_attribute(acct, "currency"))

acct.currency = "USD"
print(locate_instance_attribute(acct, "currency"))

assert acct.currency == "USD"
assert Account.currency == "EUR"

{'resolved_value': 'EUR', 'stored_on_instance': False, 'stored_on_class': True}
{'resolved_value': 'USD', 'stored_on_instance': True, 'stored_on_class': True}


## Problem 12 — The mutable class-attribute trap

The class below incorrectly shares one list among all instances:

```python
class BadCart:
    items = []
```

Demonstrate the bug with two instances, then write a corrected version using per-instance state.

In [13]:
# Solution 12

class BadCart:
    items = []

bad_a = BadCart()
bad_b = BadCart()

bad_a.items.append("book")

print("bad_a:", bad_a.items)
print("bad_b:", bad_b.items)
print("same object?", bad_a.items is bad_b.items)

assert bad_b.items == ["book"]
assert bad_a.items is bad_b.items


class GoodCart:
    def __init__(self):
        self.items = []

good_a = GoodCart()
good_b = GoodCart()

good_a.items.append("book")

print("good_a:", good_a.items)
print("good_b:", good_b.items)
print("same object?", good_a.items is good_b.items)

assert good_a.items == ["book"]
assert good_b.items == []
assert good_a.items is not good_b.items

bad_a: ['book']
bad_b: ['book']
same object? True
good_a: ['book']
good_b: []
same object? False


### Best-practice note

Use a class attribute for data that is intentionally shared across all instances. Use instance attributes for per-object mutable state.

## Problem 13 — Intentional shared state

Not every mutable class attribute is a bug. Build a class-level registry that records every unique event type registered by any instance.

Make the shared nature explicit with a class method.

In [14]:
# Solution 13

class EventRegistry:
    event_types = set()

    @classmethod
    def register(cls, event_type):
        cls.event_types.add(event_type)

first = EventRegistry()
second = EventRegistry()

first.register("created")
second.register("deleted")
second.register("created")

print(EventRegistry.event_types)

assert EventRegistry.event_types == {"created", "deleted"}
assert first.event_types is EventRegistry.event_types
assert second.event_types is EventRegistry.event_types

{'created', 'deleted'}


## Problem 14 — Fix accidental shadowing inside an instance method

The following method appears to update a shared counter, but actually creates an instance attribute:

```python
class Worker:
    processed = 0

    def process(self):
        self.processed += 1
```

Demonstrate the problem and provide two corrected designs.

In [15]:
# Solution 14

class BadWorker:
    processed = 0

    def process(self):
        self.processed += 1

w1 = BadWorker()
w2 = BadWorker()

w1.process()
w2.process()

print("BadWorker.processed:", BadWorker.processed)
print("w1.processed:", w1.processed)
print("w2.processed:", w2.processed)

assert BadWorker.processed == 0


# Fix A: explicitly update the defining class
class WorkerA:
    processed = 0

    def process(self):
        WorkerA.processed += 1


# Fix B: use a classmethod so subclass-aware shared state is possible
class WorkerB:
    processed = 0

    @classmethod
    def record_processed(cls):
        cls.processed += 1

    def process(self):
        type(self).record_processed()


a1, a2 = WorkerA(), WorkerA()
a1.process()
a2.process()

b1, b2 = WorkerB(), WorkerB()
b1.process()
b2.process()

print("WorkerA.processed:", WorkerA.processed)
print("WorkerB.processed:", WorkerB.processed)

assert WorkerA.processed == 2
assert WorkerB.processed == 2

BadWorker.processed: 0
w1.processed: 1
w2.processed: 1
WorkerA.processed: 2
WorkerB.processed: 2


# Part III — Inheritance and Namespace Resolution

## Problem 15 — Inherited class attributes are not local namespace entries

Create a base class with `timeout = 30` and a subclass with no `timeout` entry.

Show that:

- `Subclass.timeout` works
- `"timeout" in Subclass.__dict__` is false
- `"timeout" in Base.__dict__` is true

In [16]:
# Solution 15

class BaseClient:
    timeout = 30

class ApiClient(BaseClient):
    pass

print(ApiClient.timeout)
print("timeout" in ApiClient.__dict__)
print("timeout" in BaseClient.__dict__)

assert ApiClient.timeout == 30
assert "timeout" not in ApiClient.__dict__
assert "timeout" in BaseClient.__dict__

30
False
True


## Problem 16 — Override without changing the base class

Override an inherited class attribute on a subclass. Then mutate the base-class attribute and prove that the subclass keeps its own override.

In [17]:
# Solution 16

class BaseSerializer:
    format = "json"

class CsvSerializer(BaseSerializer):
    format = "csv"

print(BaseSerializer.format, CsvSerializer.format)

BaseSerializer.format = "msgpack"

print(BaseSerializer.format, CsvSerializer.format)

assert BaseSerializer.format == "msgpack"
assert CsvSerializer.format == "csv"
assert CsvSerializer.__dict__["format"] == "csv"

json csv
msgpack csv


## Problem 17 — Delete a subclass override to reveal the base value

Start with a subclass override. Delete the override from the subclass. Verify that the inherited base attribute becomes visible again.

In [18]:
# Solution 17

class ParentPolicy:
    retries = 3

class ChildPolicy(ParentPolicy):
    retries = 10

print("before:", ChildPolicy.retries)

del ChildPolicy.retries

print("after:", ChildPolicy.retries)
print("local?", "retries" in ChildPolicy.__dict__)

assert ChildPolicy.retries == 3
assert "retries" not in ChildPolicy.__dict__

before: 10
after: 3
local? False


## Problem 18 — Base-class runtime mutation propagates only when not shadowed

Create one subclass that inherits a class attribute and another that overrides it. Change the base attribute dynamically and compare results.

In [19]:
# Solution 18

class BaseFeature:
    enabled = False

class InheritingFeature(BaseFeature):
    pass

class OverridingFeature(BaseFeature):
    enabled = True

BaseFeature.enabled = "base-updated"

print("base:", BaseFeature.enabled)
print("inheriting:", InheritingFeature.enabled)
print("overriding:", OverridingFeature.enabled)

assert InheritingFeature.enabled == "base-updated"
assert OverridingFeature.enabled is True

base: base-updated
inheriting: base-updated
overriding: True


## Problem 19 — Locate the defining class in the MRO

Write `find_defining_class(cls, name)` that searches `cls.__mro__` and returns the first class whose own `__dict__` contains `name`.

Test it with overridden and inherited attributes.

In [20]:
# Solution 19

def find_defining_class(cls, name):
    for candidate in cls.__mro__:
        if name in candidate.__dict__:
            return candidate
    return None

class Root:
    root_value = 1
    shared = "root"

class Middle(Root):
    middle_value = 2

class Leaf(Middle):
    shared = "leaf"

for attr in ["root_value", "middle_value", "shared", "missing"]:
    owner = find_defining_class(Leaf, attr)
    print(attr, "->", None if owner is None else owner.__name__)

assert find_defining_class(Leaf, "root_value") is Root
assert find_defining_class(Leaf, "middle_value") is Middle
assert find_defining_class(Leaf, "shared") is Leaf
assert find_defining_class(Leaf, "missing") is None

root_value -> Root
middle_value -> Middle
shared -> Leaf
missing -> None


## Problem 20 — Multiple inheritance and first-match lookup

Construct two parents that define the same class attribute and two children with reversed base order.

Use both resolved attribute access and `__mro__` to explain the result.

In [21]:
# Solution 20

class Left:
    source = "left"

class Right:
    source = "right"

class LeftFirst(Left, Right):
    pass

class RightFirst(Right, Left):
    pass

print("LeftFirst.source:", LeftFirst.source)
print("LeftFirst MRO:", [cls.__name__ for cls in LeftFirst.__mro__])

print("RightFirst.source:", RightFirst.source)
print("RightFirst MRO:", [cls.__name__ for cls in RightFirst.__mro__])

assert LeftFirst.source == "left"
assert RightFirst.source == "right"

LeftFirst.source: left
LeftFirst MRO: ['LeftFirst', 'Left', 'Right', 'object']
RightFirst.source: right
RightFirst MRO: ['RightFirst', 'Right', 'Left', 'object']


## Problem 21 — A subclass-aware counter: subtle class attribute behavior

Use a class method that performs `cls.count += 1`.

Predict whether subclasses share the base counter forever or eventually acquire their own `count` entry.

In [22]:
# Solution 21

class Job:
    count = 0

    @classmethod
    def record(cls):
        cls.count += 1

class ImportJob(Job):
    pass

class ExportJob(Job):
    pass

print("initial:", Job.count, ImportJob.count, ExportJob.count)

ImportJob.record()
ImportJob.record()
ExportJob.record()

print("final:", Job.count, ImportJob.count, ExportJob.count)
print("ImportJob local count?", "count" in ImportJob.__dict__)
print("ExportJob local count?", "count" in ExportJob.__dict__)

assert Job.count == 0
assert ImportJob.count == 2
assert ExportJob.count == 1
assert "count" in ImportJob.__dict__
assert "count" in ExportJob.__dict__

initial: 0 0 0
final: 0 2 1
ImportJob local count? True
ExportJob local count? True


### Solution explanation

`cls.count += 1` first reads the resolved value, then assigns the new value to `cls`. For a subclass with no local `count`, the read can come from the base class, but the assignment creates a local subclass attribute.

This is a classic example where reading and writing an attribute do not necessarily act on the same namespace entry.

# Part IV — Introspection and Dynamic APIs

## Problem 22 — `vars(cls)` and `cls.__dict__`

Compare `vars(SomeClass)` with `SomeClass.__dict__`.

Check their types and whether they expose the same direct entries.

In [23]:
# Solution 22

class IntrospectionDemo:
    alpha = 1
    beta = 2

via_vars = vars(IntrospectionDemo)
via_dunder_dict = IntrospectionDemo.__dict__

print(type(via_vars))
print(type(via_dunder_dict))
print(via_vars["alpha"], via_dunder_dict["alpha"])
print(set(via_vars) == set(via_dunder_dict))

assert via_vars["beta"] == 2
assert set(via_vars) == set(via_dunder_dict)

<class 'mappingproxy'>
<class 'mappingproxy'>
1 1
True


## Problem 23 — Build a class-attribute table across the MRO

Write a function that shows, for each class in an MRO, whether a target name is stored directly there.

Then display it for an attribute that is overridden halfway down an inheritance chain.

In [24]:
# Solution 23

def mro_attribute_table(cls, name):
    rows = []
    for candidate in cls.__mro__:
        local = name in candidate.__dict__
        rows.append(
            {
                "class": candidate.__name__,
                "stored_here": local,
                "local_value": candidate.__dict__.get(name, "<not local>"),
            }
        )
    return rows

class A:
    level = "A"

class B(A):
    pass

class C(B):
    level = "C"

class D(C):
    pass

for row in mro_attribute_table(D, "level"):
    print(row)

assert D.level == "C"
assert find_defining_class(D, "level") is C

{'class': 'D', 'stored_here': False, 'local_value': '<not local>'}
{'class': 'C', 'stored_here': True, 'local_value': 'C'}
{'class': 'B', 'stored_here': False, 'local_value': '<not local>'}
{'class': 'A', 'stored_here': True, 'local_value': 'A'}
{'class': 'object', 'stored_here': False, 'local_value': '<not local>'}


## Problem 24 — Attribute existence is not the same as truthiness

A class may deliberately store falsey values such as `0`, `False`, `None`, `""`, or an empty container.

Implement a check that correctly distinguishes "attribute exists" from "resolved value is truthy".

In [25]:
# Solution 24

class Limits:
    max_retries = 0
    enabled = False
    description = ""
    fallback = None

for name in ["max_retries", "enabled", "description", "fallback", "missing"]:
    exists = hasattr(Limits, name)
    value = getattr(Limits, name, "<missing>")
    print(f"{name:12} exists={exists!s:5} value={value!r}")

assert hasattr(Limits, "max_retries")
assert not Limits.max_retries
assert not hasattr(Limits, "missing")

max_retries  exists=True  value=0
enabled      exists=True  value=False
description  exists=True  value=''
fallback     exists=True  value=None
missing      exists=False value='<missing>'


## Problem 25 — Create a reusable class-level configuration loader

Implement `apply_class_config(cls, config, allowed_names)` with these rules:

- reject unknown names
- use `setattr`
- return a dictionary of previous values
- support rolling back the mutation later

Then demonstrate apply + rollback.

In [26]:
# Solution 25

_MISSING = object()

def apply_class_config(cls, config, allowed_names):
    unknown = set(config) - set(allowed_names)
    if unknown:
        raise KeyError(f"Unknown configuration keys: {sorted(unknown)}")

    previous = {}
    for name, value in config.items():
        previous[name] = getattr(cls, name, _MISSING)
        setattr(cls, name, value)
    return previous

def rollback_class_config(cls, previous):
    for name, old_value in previous.items():
        if old_value is _MISSING:
            delattr(cls, name)
        else:
            setattr(cls, name, old_value)

class EngineConfig:
    threads = 2
    mode = "safe"

old = apply_class_config(
    EngineConfig,
    {"threads": 8, "trace": True},
    allowed_names={"threads", "mode", "trace"},
)

print("mutated:", EngineConfig.threads, EngineConfig.trace)

rollback_class_config(EngineConfig, old)

print("rolled back:", EngineConfig.threads)
print("trace exists?", hasattr(EngineConfig, "trace"))

assert EngineConfig.threads == 2
assert not hasattr(EngineConfig, "trace")

mutated: 8 True
rolled back: 2
trace exists? False


## Problem 26 — Snapshot only user-facing class data

Class namespaces also contain implementation-related entries such as `__module__`, `__dict__`, `__weakref__`, and `__doc__`.

Write `public_class_attributes(cls)` that returns only direct, non-callable, non-dunder attributes.

In [27]:
# Solution 26

def public_class_attributes(cls):
    result = {}
    for name, value in cls.__dict__.items():
        if name.startswith("__") and name.endswith("__"):
            continue
        if callable(value):
            continue
        result[name] = value
    return result

class Product:
    category = "hardware"
    tax_rate = 0.2

    def price_with_tax(self, price):
        return price * (1 + self.tax_rate)

print(public_class_attributes(Product))

assert public_class_attributes(Product) == {
    "category": "hardware",
    "tax_rate": 0.2,
}

{'category': 'hardware', 'tax_rate': 0.2}


# Part V — Methods Are Also Class Attributes

## Problem 27 — A function stored in a class namespace behaves differently through class and instance access

Define an ordinary method and inspect it:

- directly in `Class.__dict__`
- through `Class.method`
- through `instance.method`

Compare types and call behavior.

In [28]:
# Solution 27

class Greeter:
    prefix = "Hello"

    def greet(self, name):
        return f"{self.prefix}, {name}!"

g = Greeter()

raw = Greeter.__dict__["greet"]
via_class = Greeter.greet
via_instance = g.greet

print("raw type:", type(raw))
print("via class type:", type(via_class))
print("via instance type:", type(via_instance))

print(raw(g, "Ada"))
print(via_class(g, "Grace"))
print(via_instance("Linus"))

assert raw(g, "Ada") == "Hello, Ada!"
assert via_instance("Linus") == "Hello, Linus!"

raw type: <class 'function'>
via class type: <class 'function'>
via instance type: <class 'method'>
Hello, Ada!
Hello, Grace!
Hello, Linus!


### Why this belongs in a class-attributes lesson

The function object is stored in the class namespace as an attribute. Python's attribute-access machinery can transform how it behaves when accessed through an instance. This is one reason direct namespace inspection and ordinary attribute lookup are not identical operations.

## Problem 28 — Monkey-patch a method at runtime

Replace a class method implementation at runtime by assigning a new function to the class.

Show that existing instances use the new behavior immediately.

In [29]:
# Solution 28

class Formatter:
    def format(self, text):
        return text.lower()

f1 = Formatter()
f2 = Formatter()

print(f1.format("Hello"))

def loud_format(self, text):
    return text.upper() + "!"

Formatter.format = loud_format

print(f1.format("Hello"))
print(f2.format("World"))

assert f1.format("Hello") == "HELLO!"
assert f2.format("World") == "WORLD!"

hello
HELLO!
WORLD!


### Best-practice note

Runtime monkey-patching can be useful in testing or specialized frameworks, but it makes program behavior harder to reason about. Prefer explicit dependency injection or subclassing in ordinary application code.

## Problem 29 — Dynamically attach a method with `setattr`

Generate a function, attach it to a class under a dynamic name, and call it from an instance.

Then inspect whether the name appears in the class namespace.

In [30]:
# Solution 29

class Calculator:
    factor = 3

def multiply(self, value):
    return self.factor * value

method_name = "scale"
setattr(Calculator, method_name, multiply)

calc = Calculator()

print(calc.scale(10))
print("scale" in Calculator.__dict__)
print(Calculator.__dict__["scale"])

assert calc.scale(10) == 30
assert "scale" in Calculator.__dict__

30
True
<function multiply at 0x00000137E21820C0>


# Part VI — Advanced Design Problems

## Problem 30 — Class attribute validation with `__init_subclass__`

Build a base class that requires every subclass to define a direct class attribute named `kind`.

Important distinction: checking `hasattr(cls, "kind")` would allow an inherited value. The requirement here is that **each subclass must define its own** `kind`, so inspect `cls.__dict__`.

In [31]:
# Solution 30

class Plugin:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if "kind" not in cls.__dict__:
            raise TypeError(
                f"{cls.__name__} must define its own class attribute 'kind'"
            )

class JsonPlugin(Plugin):
    kind = "json"

print(JsonPlugin.kind)

try:
    class BrokenPlugin(JsonPlugin):
        pass
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

class CsvPlugin(JsonPlugin):
    kind = "csv"

assert JsonPlugin.kind == "json"
assert CsvPlugin.kind == "csv"

json
TypeError: BrokenPlugin must define its own class attribute 'kind'


## Problem 31 — Per-subclass registries without accidental sharing

Create a base class that gives each subclass its own `registry` list.

Use `__init_subclass__` to ensure every new subclass starts with a fresh list rather than inheriting and sharing the same mutable object.

In [32]:
# Solution 31

class HandlerBase:
    registry = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.registry = []

    @classmethod
    def register(cls, name):
        cls.registry.append(name)

class FileHandler(HandlerBase):
    pass

class NetworkHandler(HandlerBase):
    pass

FileHandler.register("txt")
FileHandler.register("csv")
NetworkHandler.register("http")

print("base:", HandlerBase.registry)
print("file:", FileHandler.registry)
print("network:", NetworkHandler.registry)

assert HandlerBase.registry == []
assert FileHandler.registry == ["txt", "csv"]
assert NetworkHandler.registry == ["http"]
assert FileHandler.registry is not NetworkHandler.registry

base: []
file: ['txt', 'csv']
network: ['http']


## Problem 32 — A class-level instance counter with inheritance semantics

Design two counter strategies:

1. one total shared by the whole family
2. one separate count per concrete subclass

Instantiate several objects and compare the results.

In [33]:
# Solution 32

# Strategy A: one family-wide total
class SharedCounterBase:
    total_created = 0

    def __init__(self):
        SharedCounterBase.total_created += 1

class SharedA(SharedCounterBase):
    pass

class SharedB(SharedCounterBase):
    pass

SharedA()
SharedA()
SharedB()

print("shared total:", SharedCounterBase.total_created)


# Strategy B: separate counter per concrete subclass
class PerSubclassCounterBase:
    total_created = 0

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.total_created = 0

    def __init__(self):
        type(self).total_created += 1

class SeparateA(PerSubclassCounterBase):
    pass

class SeparateB(PerSubclassCounterBase):
    pass

SeparateA()
SeparateA()
SeparateB()

print("SeparateA:", SeparateA.total_created)
print("SeparateB:", SeparateB.total_created)
print("base:", PerSubclassCounterBase.total_created)

assert SharedCounterBase.total_created == 3
assert SeparateA.total_created == 2
assert SeparateB.total_created == 1
assert PerSubclassCounterBase.total_created == 0

shared total: 3
SeparateA: 2
SeparateB: 1
base: 0


## Problem 33 — Transactional temporary class mutation

Create a context manager that temporarily changes one or more class attributes and then restores the original class state—even if an exception occurs inside the `with` block.

It must correctly handle attributes that did not exist before the context.

In [34]:
# Solution 33

from contextlib import contextmanager

_SENTINEL = object()

@contextmanager
def temporary_class_attributes(cls, **changes):
    previous = {
        name: getattr(cls, name, _SENTINEL)
        for name in changes
    }

    try:
        for name, value in changes.items():
            setattr(cls, name, value)
        yield cls
    finally:
        for name, old_value in previous.items():
            if old_value is _SENTINEL:
                delattr(cls, name)
            else:
                setattr(cls, name, old_value)

class RequestConfig:
    timeout = 10
    retries = 2

print("before:", RequestConfig.timeout, hasattr(RequestConfig, "debug"))

try:
    with temporary_class_attributes(
        RequestConfig,
        timeout=1,
        debug=True,
    ):
        print("inside:", RequestConfig.timeout, RequestConfig.debug)
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

print("after:", RequestConfig.timeout, hasattr(RequestConfig, "debug"))

assert RequestConfig.timeout == 10
assert not hasattr(RequestConfig, "debug")

before: 10 False
inside: 1 True
after: 10 False


## Problem 34 — Find local, inherited, or missing class attributes

Implement `class_attribute_origin(cls, name)` that returns one of:

- `"local"` — stored directly in `cls.__dict__`
- `"inherited:<ClassName>"` — found on another class in the MRO
- `"resolved-but-not-in-class-mro-dicts"` — accessible but not found in inspected class MRO mappings
- `"missing"`

Test it on user-defined attributes, inherited attributes, and `__name__`.

In [35]:
# Solution 34

def class_attribute_origin(cls, name):
    if name in cls.__dict__:
        return "local"

    for candidate in cls.__mro__[1:]:
        if name in candidate.__dict__:
            return f"inherited:{candidate.__name__}"

    if hasattr(cls, name):
        return "resolved-but-not-in-class-mro-dicts"

    return "missing"

class OriginBase:
    inherited_value = 42

class OriginChild(OriginBase):
    local_value = 99

for name in ["local_value", "inherited_value", "__name__", "totally_missing"]:
    print(name, "->", class_attribute_origin(OriginChild, name))

assert class_attribute_origin(OriginChild, "local_value") == "local"
assert class_attribute_origin(OriginChild, "inherited_value") == "inherited:OriginBase"
assert class_attribute_origin(OriginChild, "totally_missing") == "missing"

local_value -> local
inherited_value -> inherited:OriginBase
__name__ -> resolved-but-not-in-class-mro-dicts
totally_missing -> missing


# Part VII — Debugging Challenges

## Problem 35 — Debug a shared-default bug

A developer expects every `Session` to have an independent `headers` dictionary:

```python
class Session:
    headers = {"User-Agent": "MyApp"}
```

One instance adds an authorization header and unexpectedly changes another instance.

Reproduce the bug, then fix it while retaining a class-level immutable default template.

In [36]:
# Solution 35

class BadSession:
    headers = {"User-Agent": "MyApp"}

s1 = BadSession()
s2 = BadSession()

s1.headers["Authorization"] = "Bearer SECRET"

print("s2 unexpectedly sees:", s2.headers)
assert "Authorization" in s2.headers


class Session:
    DEFAULT_HEADERS = {"User-Agent": "MyApp"}

    def __init__(self):
        # Copy shared template into per-instance mutable state.
        self.headers = self.DEFAULT_HEADERS.copy()

a = Session()
b = Session()

a.headers["Authorization"] = "Bearer SECRET"

print("a:", a.headers)
print("b:", b.headers)
print("template:", Session.DEFAULT_HEADERS)

assert "Authorization" in a.headers
assert "Authorization" not in b.headers
assert "Authorization" not in Session.DEFAULT_HEADERS

s2 unexpectedly sees: {'User-Agent': 'MyApp', 'Authorization': 'Bearer SECRET'}
a: {'User-Agent': 'MyApp', 'Authorization': 'Bearer SECRET'}
b: {'User-Agent': 'MyApp'}
template: {'User-Agent': 'MyApp'}


## Problem 36 — Debug an inheritance configuration leak

A base class stores a mutable `options` dictionary. Two subclasses mutate it and unexpectedly affect each other.

Fix the design so each subclass owns an independent dictionary.

In [37]:
# Solution 36

class BadProcessor:
    options = {"verbose": False}

class BadCsv(BadProcessor):
    pass

class BadJson(BadProcessor):
    pass

BadCsv.options["delimiter"] = ","

print("BadJson leaked options:", BadJson.options)
assert "delimiter" in BadJson.options


class Processor:
    options = {"verbose": False}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        # Fresh copy for every direct/indirect subclass.
        cls.options = dict(cls.options)

class CsvProcessor(Processor):
    pass

class JsonProcessor(Processor):
    pass

CsvProcessor.options["delimiter"] = ","
JsonProcessor.options["indent"] = 2

print("CsvProcessor:", CsvProcessor.options)
print("JsonProcessor:", JsonProcessor.options)

assert "indent" not in CsvProcessor.options
assert "delimiter" not in JsonProcessor.options
assert CsvProcessor.options is not JsonProcessor.options

BadJson leaked options: {'verbose': False, 'delimiter': ','}
CsvProcessor: {'verbose': False, 'delimiter': ','}
JsonProcessor: {'verbose': False, 'indent': 2}


## Problem 37 — Dynamic class patch with verification

Write `patch_class(cls, **changes)` that:

1. refuses to patch any dunder name
2. records whether each attribute existed directly, was inherited, or was absent
3. applies changes
4. returns a report

Test it on one local attribute, one inherited attribute, and one new attribute.

In [38]:
# Solution 37

def patch_class(cls, **changes):
    report = {}

    for name in changes:
        if name.startswith("__") and name.endswith("__"):
            raise ValueError(f"Refusing to patch dunder name: {name}")

    for name, value in changes.items():
        if name in cls.__dict__:
            origin = "local"
        else:
            owner = find_defining_class(cls, name)
            origin = (
                f"inherited:{owner.__name__}"
                if owner is not None
                else "absent"
            )

        old_value = getattr(cls, name, _MISSING)
        setattr(cls, name, value)

        report[name] = {
            "origin_before": origin,
            "old_value": "<missing>" if old_value is _MISSING else old_value,
            "new_value": value,
            "local_after": name in cls.__dict__,
        }

    return report

class PatchBase:
    inherited = 1

class PatchTarget(PatchBase):
    local = 2

report = patch_class(
    PatchTarget,
    inherited=10,
    local=20,
    new_name=30,
)

for name, details in report.items():
    print(name, "->", details)

assert PatchTarget.inherited == 10
assert PatchTarget.local == 20
assert PatchTarget.new_name == 30
assert "inherited" in PatchTarget.__dict__  # now shadows base

inherited -> {'origin_before': 'inherited:PatchBase', 'old_value': 1, 'new_value': 10, 'local_after': True}
local -> {'origin_before': 'local', 'old_value': 2, 'new_value': 20, 'local_after': True}
new_name -> {'origin_before': 'absent', 'old_value': '<missing>', 'new_value': 30, 'local_after': True}


# Part VIII — Capstone Problems

## Problem 38 — Build a class configuration audit report

Write `audit_class(cls)` that reports:

- the class name
- its MRO names
- direct public data attributes
- direct callable attributes
- inherited public names visible through the MRO but not stored locally

Use only Python's built-in introspection facilities.

In [39]:
# Solution 38

def audit_class(cls):
    local_public_data = {}
    local_public_callables = {}

    for name, value in cls.__dict__.items():
        if name.startswith("_"):
            continue
        if callable(value):
            local_public_callables[name] = value
        else:
            local_public_data[name] = value

    inherited_public = {}

    for base in cls.__mro__[1:]:
        for name, value in base.__dict__.items():
            if name.startswith("_"):
                continue
            if name not in cls.__dict__ and name not in inherited_public:
                inherited_public[name] = {
                    "from": base.__name__,
                    "value": value,
                }

    return {
        "class": cls.__name__,
        "mro": [c.__name__ for c in cls.__mro__],
        "local_public_data": local_public_data,
        "local_public_callables": sorted(local_public_callables),
        "inherited_public": inherited_public,
    }

class AuditBase:
    region = "global"

    def base_method(self):
        return "base"

class AuditTarget(AuditBase):
    mode = "fast"

    def run(self):
        return "running"

audit = audit_class(AuditTarget)

for key, value in audit.items():
    print(key, "=", value)

assert audit["class"] == "AuditTarget"
assert audit["local_public_data"]["mode"] == "fast"
assert audit["inherited_public"]["region"]["from"] == "AuditBase"

class = AuditTarget
mro = ['AuditTarget', 'AuditBase', 'object']
local_public_data = {'mode': 'fast'}
local_public_callables = ['run']
inherited_public = {'region': {'from': 'AuditBase', 'value': 'global'}, 'base_method': {'from': 'AuditBase', 'value': <function AuditBase.base_method at 0x00000137E2183380>}}


## Problem 39 — Design a class-based feature flag system

Requirements:

- flags are class attributes
- unknown flags raise `KeyError`
- inherited flags may be read
- setting a flag on a subclass must create or replace a local override without changing the base
- deleting a subclass override should reveal the inherited base value again

Implement `get_flag`, `set_flag`, and `reset_flag`.

In [40]:
# Solution 39

class FeatureFlagBase:
    dark_mode = False
    beta_api = False

    @classmethod
    def get_flag(cls, name):
        if not hasattr(cls, name):
            raise KeyError(name)
        value = getattr(cls, name)
        if not isinstance(value, bool):
            raise TypeError(f"{name!r} is not a boolean flag")
        return value

    @classmethod
    def set_flag(cls, name, value):
        if not isinstance(value, bool):
            raise TypeError("Feature flag values must be bool")
        if not hasattr(cls, name):
            raise KeyError(name)
        setattr(cls, name, value)

    @classmethod
    def reset_flag(cls, name):
        if name in cls.__dict__:
            delattr(cls, name)
        if not hasattr(cls, name):
            raise KeyError(name)

class EuropeanFlags(FeatureFlagBase):
    pass

print("inherited:", EuropeanFlags.get_flag("dark_mode"))

EuropeanFlags.set_flag("dark_mode", True)

print("base:", FeatureFlagBase.dark_mode)
print("subclass override:", EuropeanFlags.dark_mode)
print("local?", "dark_mode" in EuropeanFlags.__dict__)

EuropeanFlags.reset_flag("dark_mode")

print("after reset:", EuropeanFlags.dark_mode)
print("local?", "dark_mode" in EuropeanFlags.__dict__)

assert FeatureFlagBase.dark_mode is False
assert EuropeanFlags.dark_mode is False
assert "dark_mode" not in EuropeanFlags.__dict__

inherited: False
base: False
subclass override: True
local? True
after reset: False
local? False


## Problem 40 — Final capstone: reversible plugin registration

Create a plugin system in which:

- plugin classes define a class attribute `kind`
- a registry maps `kind -> plugin class`
- registration is performed with a class method
- duplicate kinds are rejected
- plugins can be unregistered
- the registry is shared intentionally by the plugin family
- the registry itself is never exposed for mutation; callers receive a copy

Then test all behaviors.

In [41]:
# Solution 40

class PluginBase:
    _registry = {}

    kind = None

    @classmethod
    def register(cls, plugin_cls):
        if not issubclass(plugin_cls, cls):
            raise TypeError("plugin_cls must be a subclass of the registry owner")

        kind = getattr(plugin_cls, "kind", None)
        if not isinstance(kind, str) or not kind:
            raise ValueError("Plugin must define a non-empty string class attribute 'kind'")

        if kind in cls._registry:
            raise KeyError(f"Duplicate plugin kind: {kind!r}")

        cls._registry[kind] = plugin_cls

    @classmethod
    def unregister(cls, kind):
        try:
            return cls._registry.pop(kind)
        except KeyError:
            raise KeyError(f"Unknown plugin kind: {kind!r}") from None

    @classmethod
    def resolve(cls, kind):
        try:
            return cls._registry[kind]
        except KeyError:
            raise KeyError(f"Unknown plugin kind: {kind!r}") from None

    @classmethod
    def registry_snapshot(cls):
        return dict(cls._registry)


class JsonOutput(PluginBase):
    kind = "json"

class CsvOutput(PluginBase):
    kind = "csv"


PluginBase.register(JsonOutput)
PluginBase.register(CsvOutput)

print(PluginBase.registry_snapshot())
print(PluginBase.resolve("json").__name__)

snapshot = PluginBase.registry_snapshot()
snapshot["fake"] = object

print("fake leaked into real registry?", "fake" in PluginBase.registry_snapshot())

removed = PluginBase.unregister("csv")
print("removed:", removed.__name__)
print("remaining:", PluginBase.registry_snapshot())

assert PluginBase.resolve("json") is JsonOutput
assert "fake" not in PluginBase.registry_snapshot()
assert removed is CsvOutput
assert "csv" not in PluginBase.registry_snapshot()

try:
    PluginBase.register(JsonOutput)
except KeyError as exc:
    print(type(exc).__name__ + ":", exc)

{'json': <class '__main__.JsonOutput'>, 'csv': <class '__main__.CsvOutput'>}
JsonOutput
fake leaked into real registry? False
removed: CsvOutput
remaining: {'json': <class '__main__.JsonOutput'>}
KeyError: "Duplicate plugin kind: 'json'"


# Extra Drill Set — Short Advanced Exercises

Try these **before** looking at the compact solutions below.

1. Write a one-liner that checks whether an attribute is stored *directly* on a class.
2. Write a one-liner that safely reads a possibly missing dynamic attribute.
3. Explain why `cls.__dict__[name]` and `getattr(cls, name)` may produce different outcomes.
4. Show how to create a class attribute whose name is stored in a string.
5. Show how to remove a class attribute whose name is stored in a string.
6. Demonstrate that a subclass can read a base attribute without storing it locally.
7. Demonstrate that assigning that same name to the subclass creates a shadow.
8. Demonstrate that deleting the subclass shadow reveals the base value.
9. Explain why a mutable list used as a class attribute is often dangerous.
10. Give one case where a mutable class attribute is intentionally useful.
11. Write code that checks which class in the MRO directly defines an attribute.
12. Explain why `hasattr(cls, "__name__")` and `"__name__" in cls.__dict__` can disagree.
13. Explain why a saved `mappingproxy` reflects later legal mutations.
14. Show why `instance.x += 1` can create instance state even if `x` started as a class attribute.
15. Explain why `cls.x += 1` in a class method may create a subclass-local attribute.

In [42]:
# Compact solutions to the Extra Drill Set

# 1
class DirectDemo:
    x = 1
print("1:", "x" in DirectDemo.__dict__)

# 2
print("2:", getattr(DirectDemo, "missing", None))

# 3
class ParentDemo:
    inherited = 7
class ChildDemo(ParentDemo):
    pass
print("3:", "inherited" in ChildDemo.__dict__, getattr(ChildDemo, "inherited"))

# 4
setattr(DirectDemo, "dynamic_name", 123)
print("4:", DirectDemo.dynamic_name)

# 5
delattr(DirectDemo, "dynamic_name")
print("5:", hasattr(DirectDemo, "dynamic_name"))

# 6
print("6:", ChildDemo.inherited, "inherited" in ChildDemo.__dict__)

# 7
ChildDemo.inherited = 99
print("7:", ChildDemo.inherited, ParentDemo.inherited)

# 8
del ChildDemo.inherited
print("8:", ChildDemo.inherited)

# 9 / 10
class SharedListDemo:
    intentionally_shared = []
print("9/10:", SharedListDemo.intentionally_shared)

# 11
print("11:", find_defining_class(ChildDemo, "inherited").__name__)

# 12
print("12:", hasattr(DirectDemo, "__name__"), "__name__" in DirectDemo.__dict__)

# 13
proxy = DirectDemo.__dict__
DirectDemo.after_proxy = "visible"
print("13:", proxy["after_proxy"])

# 14
class IncrementDemo:
    x = 0
obj = IncrementDemo()
obj.x += 1
print("14:", IncrementDemo.x, obj.x, obj.__dict__)

# 15
class CounterBase:
    x = 0
    @classmethod
    def bump(cls):
        cls.x += 1

class CounterChild(CounterBase):
    pass

CounterChild.bump()
print("15:", CounterBase.x, CounterChild.x, CounterChild.__dict__["x"])

1: True
2: None
3: False 7
4: 123
5: False
6: 7 False
7: 99 7
8: 7
9/10: []
11: ParentDemo
12: True False
13: visible
14: 0 1 {'x': 1}
15: 0 1 1


# Final Review Checklist

You should now be able to reason about all of the following without guessing:

- **Where is the attribute stored?**
- **Is it direct, inherited, instance-level, or supplied by broader class-object behavior?**
- **Will assignment mutate an existing shared value or create a new shadow?**
- **Will deletion remove the attribute completely or reveal an inherited value?**
- **Is a mutable class attribute intentionally shared or accidentally shared?**
- **Should dynamic access use dotted notation, `getattr`, `setattr`, or `delattr`?**
- **Why is `Class.__dict__` useful even though it cannot be directly mutated?**
- **How does the MRO affect class-attribute resolution?**
- **How do class methods interact with inherited class attributes?**
- **When is runtime class mutation a good design, and when should it be avoided?**

## Suggested practice mode

Re-run the notebook after hiding solution cells. For every problem, write down:

1. the namespace you expect Python to inspect,
2. the value you predict,
3. whether any assignment creates a new attribute,
4. whether the change is shared, local, inherited, or shadowing.

That reasoning habit is more valuable than memorizing outputs.